In [1]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install wordcloud

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.2/555.2 kB 6.0 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [14]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
from wordcloud import WordCloud
from collections import Counter

In [15]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean up column names that may include extra spaces/newlines.
    Keeps the important ones like 'MeSH terms (Descriptor)' intact.
    """
    df = df.copy()
    df.columns = [
        re.sub(r"\s+", " ", str(c)).strip()  # collapse whitespace
        for c in df.columns
    ]
    return df

In [16]:
def create_circular_mask(w, h):
    """Generate a circular mask (255 inside circle, 0 outside)."""
    x, y = np.ogrid[:h, :w]
    center_x, center_y = w // 2, h // 2
    dist = np.sqrt((x - center_x) ** 2 + (y - center_y) ** 2)
    return (dist <= min(center_x, center_y)).astype(np.uint8) * 255

In [32]:
import re

def save_wordcloud_svg_clipped(wc, output_svg, width=2000, height=2000, margin_px=0):
    """
    Save WordCloud SVG with a circular clipPath so text cannot render outside the circle.

    Parameters
    ----------
    wc : WordCloud
        The fitted WordCloud object.
    output_svg : str
        Output SVG file path.
    width, height : int
        Must match the WordCloud width/height used to generate it.
    margin_px : int
        Shrinks the circle radius by this many pixels (useful to prevent touching edge).
    """
    svg = wc.to_svg()

    # Center and radius (circle fitted to canvas)
    cx, cy = width / 2, height / 2
    r = min(cx, cy) - margin_px

    clip_def = f"""
<defs>
  <clipPath id="wcCircleClip">
    <circle cx="{cx}" cy="{cy}" r="{r}" />
  </clipPath>
</defs>
"""

    # Insert <defs> right after the opening <svg ...> tag
    svg = re.sub(r"(<svg[^>]*>)", r"\1\n" + clip_def, svg, count=1)

    # Wrap all existing content in a clipped group.
    # WordCloud SVG typically has a single <g>...</g> block; we clip at the top level.
    # Insert <g clip-path="url(#wcCircleClip)"> right after </defs>
    svg = svg.replace("</defs>", "</defs>\n<g clip-path=\"url(#wcCircleClip)\">", 1)

    # Close the clipping group before </svg>
    svg = svg.replace("</svg>", "</g>\n</svg>", 1)

    with open(output_svg, "w", encoding="utf-8") as f:
        f.write(svg)


In [33]:
def generate_mesh_wordcloud(data_df, exclude_keywords, output_file="wordcloud.svg", theme="green"):
    """
    Generate MeSH-based word cloud and save as SVG.

    Parameters:
    - data_df: dataframe that contains MeSH columns
    - exclude_keywords: list[str], search keywords/synonyms to filter out
    - output_file: svg output path
    - theme: green/blue/orange
    """

    stop_words = [
        "Humans", "Female", "Male", "Adult", "Middle Aged", "Young Adult",
        "Aged", "Animals", "Surveys and Questionnaires"
    ]

    palettes = {
        "blue": ["#4E79A7", "#5E85B8", "#3E6582", "#2E4E62"],
        "orange": ["#F28E2B", "#F4A65A", "#F7C788", "#D77A1B"],
        "green": ["#70AD47", "#80BB59", "#588938", "#42662A"],
    }
    current_palette = palettes.get(theme, palettes["green"])

    # Normalize exclude_keywords
    exclude_keywords = [str(k).strip() for k in (exclude_keywords or []) if str(k).strip()]

    # Preprocess MeSH terms
    target_columns = ["MeSH terms (Descriptor)", "MeSH terms (Qualifier)"]

    all_phrases = []
    for col_name in target_columns:
        if col_name in data_df.columns:
            # Force to string; skip NaNs
            for cell in data_df[col_name].dropna():
                cell = str(cell)
                parts = [p.strip() for p in cell.split("|||") if p.strip()]
                all_phrases.extend(parts)
        else:
            print(f"Warning: Column '{col_name}' not found. Skipping.")

    if not all_phrases:
        print("Error: No MeSH text found in the specified columns.")
        return

    selected_freq = Counter(all_phrases)

    # Remove stop words (exact match)
    freq_clean_stop_words = {
        word: count for word, count in selected_freq.items()
        if word not in stop_words
    }

    # Remove search keywords (substring match, case-insensitive)
    freq_clean = {
        word: count for word, count in freq_clean_stop_words.items()
        if not any(bad.lower() in word.lower() for bad in exclude_keywords)
    }

    if not freq_clean:
        print("Warning: No words left after filtering. Nothing to plot.")
        return

    # WordCloud generation
    # plot
    W, H = 2000, 2000

    raw_mask = create_circular_mask(W, H)  # 255 inside circle, 0 outside
    mask = 255 - raw_mask                 # 0 inside circle (drawable), 255 outside (blocked)

    cmap = mcolors.LinearSegmentedColormap.from_list('custom_palette', current_palette)
    wc = WordCloud(
        width=W,
        height=H,
        background_color='white',
        mask=mask,
        colormap=cmap,
        collocations=False
    ).generate_from_frequencies(freq_clean)

    save_wordcloud_svg_clipped(wc, output_file, width=W, height=H, margin_px=10)
    """
    svg_data = wc.to_svg()
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(svg_data)
        print(f"Success: Saved wordcloud as '{output_file}' using theme '{theme}'")
    except IOError as e:
        print(f"Error saving file: {e}")
    """
    # import matplotlib.pyplot as plt

    # plt.figure(figsize=(5, 5))
    # plt.imshow(mask, cmap="gray")
    # plt.axis("off")
    # plt.show()
    
    # plt.figure(figsize=(6, 6))
    # plt.imshow(wc.to_image())
    # plt.axis("off")
    # plt.show()

In [36]:
# Update these paths to wherever your files actually are
base_folder = "/fs/scratch/PCON0100/feng1426/projects/playground/mprint_ranking"
maternal_csv = os.path.join(base_folder, "maternal_database_with_scores.csv")

# Read CSVs (comma-separated). Use engine='python' for safety with odd formatting.
maternal_df = pd.read_csv(maternal_csv, engine="python")

# Clean column names (important for finding 'MeSH terms (Descriptor)' reliably)
maternal_df = normalize_columns(maternal_df)


In [35]:
base_folder = "/fs/scratch/PCON0100/feng1426/projects/playground/mprint_ranking"
pediatric_csv = os.path.join(base_folder, "pediatric_database_with_scores.csv")

# Read CSVs (comma-separated). Use engine='python' for safety with odd formatting.
pediatric_df = pd.read_csv(pediatric_csv, engine="python")

# Clean column names (important for finding 'MeSH terms (Descriptor)' reliably)
pediatric_df = normalize_columns(pediatric_df)

In [20]:
print(maternal_df)

          Position      PMID  \
0       1070-19610  30363869   
1         1001-121  31424096   
2         1001-234  31424210   
3         1001-285  31424259   
4         1001-552  31424532   
...            ...       ...   
376226  0100-11259   3005444   
376227  0100-11811   3005994   
376228  0100-13872   3008055   
376229  0100-27114   3021299   
376230  0100-28969   3023154   

                                                    Title    Year Language  \
0       Are the early childhood antecedents of men's e...     NaN   |||eng   
1       Adolescent childbirth, miscarriage, and aborti...  2021.0   |||eng   
2       Phthalate and BPA Exposure in Women and Newbor...  2019.0   |||eng   
3       Insulin-like growth factor axis in pregnancy a...  2020.0   |||eng   
4       Association Between Maternal Fluoride Exposure...     NaN   |||eng   
...                                                   ...     ...      ...   
376226  Studies of catecholestrogen metabolism during ...    1986   |

In [38]:
# Your searched keywords / synonyms (customize as needed)
my_keywords = ["infant", "clinical trial"]
# Generate two wordclouds
generate_mesh_wordcloud(
    data_df=maternal_df,
    exclude_keywords=my_keywords,
    output_file="maternal_wordcloud_green.svg",
    theme="green",
)


In [37]:
# Your searched keywords / synonyms (customize as needed)
my_keywords = ["infant", "clinical trial"]

generate_mesh_wordcloud(
    data_df=pediatric_df,
    exclude_keywords=my_keywords,
    output_file="pediatric_wordcloud_green.svg",
    theme="green",
)


In [28]:
!pip show wordcloud

Name: wordcloud
Version: 1.9.5
Summary: A little word cloud generator
Home-page: 
Author: 
Author-email: Andreas Mueller <t3kcit+wordcloud@gmail.com>
License: 
Location: /users/PCON0100/feng1426/.local/lib/python3.12/site-packages
Requires: matplotlib, numpy, pillow
Required-by: 


In [27]:
!pip show numpy

Name: numpy
Version: 1.26.4
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2023, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the names of any
       contributors may be used to endorse or promote products derived
       from this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYR